# DS-Fall - 01 | Process and Visualize Data

Notebook n?y chu?n h?a c?c ngu?n d? li?u th? v? ??ng input raw c?a DS-Fall:

`X.shape = (N, 100, 6)` v?i 100 timestep = 2 gi?y ? 50 Hz, channel order `[ax, ay, az, gx, gy, gz]`.

Quy t?c quan tr?ng:
- WEDA-FALL ?? l? 50 Hz, ch? c?n c?t window 2 gi?y.
- BITS-2 v? UMAFall g?c kho?ng 20 Hz, b?t bu?c resample l?n 50 Hz tr??c khi c?t window.
- UMAFall ch? l?y SensorTag v? tr? wrist, g?m accelerometer v? gyroscope; sensor kh?c b? lo?i.
- Scaler ch? fit tr?n train split ?? tr?nh data leakage.
- Split theo subject, kh?ng split ng?u nhi?n theo window.


In [ ]:
# Mount Google Drive nếu đang chạy trên Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

import sys
from pathlib import Path

# Cấu hình root của repo. Chỉ cần đổi dòng này nếu đặt repo ở vị trí khác.
PROJECT_ROOT = Path('/content/drive/MyDrive/ds-fall')
# Debug local nếu chạy ngoài Colab:
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQ = PROJECT_ROOT / 'requirements.txt'
if REQ.exists():
    print('Project root:', PROJECT_ROOT)
else:
    print('WARNING: requirements.txt not found. Check PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# C?i dependency ch? khi thi?u package ch?nh. Cell n?y an to?n ?? ch?y l?i.
import importlib
import subprocess
MODULE_CHECKS = [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn'), ('tensorflow', 'tensorflow')]
def _module_ok(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False
missing = [pkg for module, pkg in MODULE_CHECKS if not _module_ok(module)]
if missing and REQ.exists():
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)])

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import make_config
from src.utils.io import ensure_dir
from src.utils.seed import set_seed
from src.data.preprocessing import (
    build_all_windows,
    build_preprocessing_summary,
    normalize_with_train_scaler,
    save_processed_dataset,
)
from src.data.splitting import subject_wise_split
from src.data.visualization import (
    plot_bits_resampling,
    plot_channel_stats,
    plot_class_distribution,
    plot_dataset_distribution,
    plot_imu_window,
)

# T?o config ???ng d?n v? c?c th? m?c output.
set_seed(42)
config = make_config(PROJECT_ROOT)
config.enabled_datasets = ('weda', 'bits', 'hifd', 'umafall')
for path in [config.processed_dir, config.figures_dir / 'preprocessing', config.models_dir, config.logs_dir, config.metrics_dir]:
    ensure_dir(path)

RAW_DIR = config.raw_dir
PROCESSED_DIR = config.processed_dir
OUTPUT_DIR = config.output_dir
BITS_RAW_DIR = config.bits_raw_dir
WEDA_RAW_DIR = config.weda_raw_dir
UMAFALL_RAW_DIR = config.umafall_raw_dir

print('PROJECT_ROOT =', config.project_root)
print('RAW_DIR =', RAW_DIR)
print('Enabled datasets =', config.enabled_datasets)
for name, path in [('WEDA', WEDA_RAW_DIR), ('BITS', BITS_RAW_DIR), ('UMAFall', UMAFALL_RAW_DIR)]:
    print(f'{name}:', path, 'exists=' + str(path.exists()))


## 1. Build unified windows

Cell này đọc từng dataset có sẵn trong `data/raw`, tạo window thô chưa normalize, và giữ metadata cho từng sample. Nếu thiếu một dataset, loader chỉ warning và tiếp tục xử lý dataset còn lại.


In [ ]:
# X_raw là dữ liệu đã cắt window nhưng chưa chuẩn hóa.
# Mỗi sample phải có shape 100 x 6 theo thứ tự [ax, ay, az, gx, gy, gz].
X_raw, metadata, summary = build_all_windows(config)

print('Raw window shape:', X_raw.shape)
if len(X_raw) == 0:
    raise RuntimeError('No windows were created. Check raw dataset placement under data/raw.')
if X_raw.shape[1:] != (100, 6):
    raise ValueError(f'Invalid model input shape: {X_raw.shape}. Expected (N, 100, 6).')
if not np.isfinite(X_raw).all():
    raise ValueError('X_raw contains NaN or Inf before normalization.')

print('\nMetadata preview:')
display(metadata.head(10))

print('\nWindow counts by dataset / fall label / direction:')
count_table = (
    metadata.groupby(['dataset', 'fall_label', 'direction_label'])
    .size()
    .rename('n_windows')
    .reset_index()
    .sort_values(['dataset', 'fall_label', 'direction_label'])
)
display(count_table)


## 2. Subject-wise split and train-only normalization

Split theo subject để tránh leakage: cùng một người không được xuất hiện ở nhiều split. Sau đó fit scaler chỉ trên train split, rồi áp dụng cho toàn bộ train/val/test.


In [ ]:
# 70/15/15 subject-wise split. H?m n?y ki?m tra leakage t? ??ng.
metadata, split_subjects = subject_wise_split(metadata, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=config.seed)

print('Subject lists per split:')
print(json.dumps(split_subjects, indent=2))

split_table = (
    metadata.groupby(['dataset', 'split'])
    .agg(n_windows=('sample_id', 'count'), n_subjects=('subject_id', 'nunique'))
    .reset_index()
    .sort_values(['dataset', 'split'])
)
print('\nSplit summary:')
display(split_table)

# Fit channel-wise scaler tr?n train only: mean/std t?nh qua sample v? timestep.
X, scaler, norm_stats = normalize_with_train_scaler(X_raw, metadata)
summary.update(build_preprocessing_summary(X, metadata))
summary['normalization'] = norm_stats
summary['enabled_datasets'] = list(config.enabled_datasets)
summary['bits_preprocessing'] = '20Hz row-order uniform interpolation to 50Hz before 2-second windowing'
summary['umafall_preprocessing'] = 'WRIST SensorTag accelerometer+gyroscope, timestamp interpolation to 50Hz before 2-second windowing'

print('Normalized X shape:', X.shape)
print('NaN count:', np.isnan(X).sum(), 'Inf count:', np.isinf(X).sum())
print('\nScaler fitted on train split:')
display(pd.DataFrame({'channel': scaler['channel_names'], 'mean': scaler['mean'], 'std': scaler['std']}))


## 3. Save processed dataset

Các file này là input trực tiếp cho notebook train. `metadata.csv` có một dòng cho mỗi sample trong `X.npy`.


In [ ]:
# Lưu dữ liệu đã normalize, labels, metadata, scaler và config split.
save_processed_dataset(PROCESSED_DIR, X, metadata, scaler, split_subjects, summary)
print('Saved processed files to:', PROCESSED_DIR)
print(sorted([p.name for p in PROCESSED_DIR.iterdir() if p.is_file()]))


## 4. Raw Signal And 20 Hz Resampling Checks

M?c ti?u c?a ph?n n?y l? nh?n nhanh t?n hi?u g?c v? ki?m tra ri?ng BITS-2/UMAFall: sequence kho?ng 20 Hz ph?i ???c n?i suy l?n 50 Hz tr??c khi t?o window 100 timestep.


In [ ]:
fig_dir = config.figures_dir / 'preprocessing'

# Load m?t v?i trial g?c ?? plot minh h?a. C?c plot ???c l?u v?o outputs/figures/preprocessing.
from src.data.weda_loader import find_weda_trials, load_weda_trial
from src.data.bits_loader import find_bits_trials, parse_bits_csv, resample_bits_sequence_to_50hz
from src.data.umafall_loader import find_umafall_trials, load_umafall_trial

if WEDA_RAW_DIR.exists():
    weda_trials = find_weda_trials(WEDA_RAW_DIR)
    for label, predicate in [('weda_fall_raw', lambda t: t['activity_id'].startswith('F')), ('weda_adl_raw', lambda t: t['activity_id'].startswith('D'))]:
        trial = next((t for t in weda_trials if predicate(t)), None)
        if trial:
            seq = load_weda_trial(trial['accel_path'], trial['gyro_path'])
            if seq is not None and len(seq) >= 100:
                plot_imu_window(seq[:100], title=f'{label} | first 100 raw samples', save_path=fig_dir / f'{label}.png')

if BITS_RAW_DIR.exists():
    bits_trials = find_bits_trials(BITS_RAW_DIR)
    trial = next((t for t in bits_trials if t['class_dir'] == 'fall'), None)
    if trial:
        seq20 = parse_bits_csv(trial['csv_path'], accel_source=config.bits_accel_source)
        if seq20 is not None:
            seq50 = resample_bits_sequence_to_50hz(seq20, config.bits_original_fs, config.bits_target_fs)
            if seq50 is not None:
                plot_imu_window(seq20[:min(len(seq20), 100)], title='bits_fall_raw_20hz | before resampling', save_path=fig_dir / 'bits_fall_raw_20hz.png')
                plot_bits_resampling(seq20, seq50, save_path=fig_dir / 'bits_resampling_validation.png')
                print('BITS original shape:', seq20.shape, 'resampled shape:', seq50.shape, 'final model window length:', X.shape[1])

if UMAFALL_RAW_DIR.exists():
    umafall_trials = find_umafall_trials(UMAFALL_RAW_DIR)
    trial = next((t for t in umafall_trials if t['fall_label'] == 'fall' and t['direction_label'] in {'forward', 'backward', 'lateral'}), None)
    if trial:
        loaded = load_umafall_trial(
            trial['csv_path'],
            sensor_positions=config.umafall_sensor_positions,
            target_fs=config.umafall_target_fs,
        )
        if loaded is not None:
            seq50, trial_meta = loaded
            plot_imu_window(seq50[:100], title='umafall_wrist_fall_resampled_50hz | first model window', save_path=fig_dir / 'umafall_wrist_fall_resampled_50hz.png')
            print('UMAFall wrist trial:', trial['csv_path'])
            print('UMAFall sensor:', trial_meta['sensor_position'], 'sensor_id:', trial_meta['sensor_id'])
            print('UMAFall approx original length:', int(round(len(seq50) * config.umafall_original_fs / config.umafall_target_fs)), 'resampled length:', len(seq50))


## 5. Final Visual Report And Quality Summary

Ph?n cu?i t?o c?c plot/report ch?nh ?? ki?m tra dataset sau x? l?: shape, NaN/Inf, ph?n b? dataset, nh?n fall, nh?n direction, split, normalization v? c?c dataset 20 Hz ?? resample.


In [ ]:
# 5.1 Plot representative processed windows.
for direction in ['forward', 'backward', 'lateral']:
    idx = metadata.index[metadata['direction_label'].eq(direction)].tolist()
    if idx:
        plot_imu_window(
            X[idx[0]],
            title=f'processed_{direction}_fall | normalized 100x6 window',
            save_path=fig_dir / f'processed_{direction}_fall.png',
        )

idx = metadata.index[metadata['fall_label'].eq(0)].tolist()
if idx:
    plot_imu_window(
        X[idx[0]],
        title='processed_non_fall | normalized 100x6 window',
        save_path=fig_dir / 'processed_non_fall.png',
    )

# 5.2 Core distribution plots.
plot_dataset_distribution(metadata, save_path=fig_dir / 'windows_by_dataset.png')
plot_class_distribution(metadata, 'fall_label', 'Windows by fall label', save_path=fig_dir / 'windows_by_fall_label.png')
plot_class_distribution(metadata, 'direction_label', 'Windows by direction label', save_path=fig_dir / 'windows_by_direction_label.png')
plot_class_distribution(metadata, 'split', 'Windows by split', save_path=fig_dir / 'windows_by_split.png')
plot_channel_stats(X_raw, 'Channel stats before normalization', save_path=fig_dir / 'channel_stats_before_norm.png')
plot_channel_stats(X, 'Channel stats after normalization', save_path=fig_dir / 'channel_stats_after_norm.png')

# 5.3 Compact summary tables.
quality_table = pd.DataFrame([
    {'check': 'X shape', 'value': str(tuple(X.shape))},
    {'check': 'Expected sample shape', 'value': '(100, 6)'},
    {'check': 'NaN count', 'value': int(np.isnan(X).sum())},
    {'check': 'Inf count', 'value': int(np.isinf(X).sum())},
    {'check': 'Sampling rate model', 'value': '50 Hz'},
    {'check': 'Enabled datasets', 'value': ', '.join(config.enabled_datasets)},
    {'check': 'Window duration', 'value': '2 seconds'},
    {'check': 'Channel order', 'value': '[ax, ay, az, gx, gy, gz]'},
])

dataset_summary = (
    metadata.groupby('dataset')
    .agg(
        n_windows=('sample_id', 'count'),
        n_subjects=('subject_id', 'nunique'),
        fall_windows=('fall_label', 'sum'),
        resampled_windows=('resampled', lambda s: int(s.astype(bool).sum())),
    )
    .reset_index()
)
dataset_summary['non_fall_windows'] = dataset_summary['n_windows'] - dataset_summary['fall_windows']
dataset_summary['fall_ratio'] = (dataset_summary['fall_windows'] / dataset_summary['n_windows']).round(4)

direction_summary = (
    metadata.groupby(['direction_label', 'direction_supervised'])
    .size()
    .rename('n_windows')
    .reset_index()
    .sort_values(['direction_supervised', 'direction_label'])
)

split_summary = (
    metadata.groupby(['dataset', 'split'])
    .agg(n_windows=('sample_id', 'count'), n_subjects=('subject_id', 'nunique'))
    .reset_index()
    .sort_values(['dataset', 'split'])
)

print('Quality summary')
display(quality_table)
print('Dataset summary')
display(dataset_summary)
print('Direction supervision summary')
display(direction_summary)
print('Split summary')
display(split_summary)

for dataset_name, label in [('bits', 'BITS'), ('umafall', 'UMAFall')]:
    ds_meta = metadata[metadata['dataset'].eq(dataset_name)]
    if not ds_meta.empty and {'original_length', 'resampled_length'}.issubset(ds_meta.columns):
        print(f'{label} length summary after 20 Hz -> 50 Hz resampling')
        display(ds_meta[['original_length', 'resampled_length']].describe().round(2))

# 5.4 One-page dashboard for the processed dataset.
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
sns.countplot(data=metadata, x='dataset', ax=axes[0, 0], order=sorted(metadata['dataset'].unique()))
axes[0, 0].set_title('Windows by dataset')

fall_by_dataset = metadata.pivot_table(index='dataset', columns='fall_label', values='sample_id', aggfunc='count', fill_value=0)
fall_by_dataset = fall_by_dataset.rename(columns={0: 'non_fall', 1: 'fall'})
fall_by_dataset.plot(kind='bar', stacked=True, ax=axes[0, 1])
axes[0, 1].set_title('Fall vs non-fall by dataset')
axes[0, 1].set_xlabel('dataset')
axes[0, 1].tick_params(axis='x', rotation=0)

sns.countplot(data=metadata, x='direction_label', hue='direction_supervised', ax=axes[0, 2])
axes[0, 2].set_title('Direction labels and supervision mask')
axes[0, 2].tick_params(axis='x', rotation=25)

sns.countplot(data=metadata, x='split', hue='dataset', ax=axes[1, 0])
axes[1, 0].set_title('Windows by split')

subject_counts = metadata.groupby('dataset')['subject_id'].nunique().reset_index(name='n_subjects')
sns.barplot(data=subject_counts, x='dataset', y='n_subjects', ax=axes[1, 1])
axes[1, 1].set_title('Subjects by dataset')

resampled_counts = metadata.groupby(['dataset', 'resampled']).size().reset_index(name='n_windows')
sns.barplot(data=resampled_counts, x='dataset', y='n_windows', hue='resampled', ax=axes[1, 2])
axes[1, 2].set_title('Resampled windows')

for ax in axes.ravel():
    ax.grid(True, axis='y', alpha=0.25)
fig.suptitle('DS-Fall processed dataset overview', fontsize=16)
fig.tight_layout()
dashboard_path = fig_dir / 'processed_dataset_dashboard.png'
fig.savefig(dashboard_path, dpi=160, bbox_inches='tight')
plt.show()

print('Saved preprocessing figures to:', fig_dir)
display(pd.DataFrame({'figure': sorted([p.name for p in fig_dir.glob('*.png')])}))
